Knowledge Graph — Notebook 3
Querying, Traversal and Multi-Hop Reasoning--

The flow: 
TRAVEL QUESTION
      ↓
Understand the question
      ↓
Identify entities
      ↓
Identify relationship
      ↓
Convert question into graph pattern
      ↓
Query the Knowledge Graph
      ↓
If necessary, traverse multiple edges
      ↓
Combine results
      ↓
Reason over the graph
      ↓
Answer
In KG-03, students will identify entities and relationships manually.

We will later ask: Can a machine identify them automatically from natural language?

That will lead naturally to Information Extraction / NLP / LLM-based KG construction.

# Knowledge Graph — Notebook 3
## Querying, Traversal and Multi-Hop Reasoning

### Running Problem: Travel Planning

In Notebook 1, we understood why a Knowledge Graph is useful.

In Notebook 2, we built a small Travel Planning Knowledge Graph.

In this notebook, we will learn:

1. How to ask questions to a Knowledge Graph
2. How to convert a question into a graph pattern
3. How to query the graph
4. How to traverse the graph
5. What is multi-hop traversal?
6. How multi-hop reasoning helps in travel planning
7. How to identify entities and relationships from a question

### Learning Journey

Question
→ Graph Pattern
→ Query
→ Traversal
→ Multi-Hop
→ Reasoning
→ Answer

## Recall the Travel Knowledge Graph

Consider the following travel information:

- Chennai is located in Tamil Nadu.
- Mahabalipuram is near Chennai.
- Mahabalipuram is a heritage destination.
- Marina Beach is located in Chennai.
- Chennai is connected to Bengaluru.
- Bengaluru is connected to Mysuru.
- Hotel SeaView is located in Mahabalipuram.
- Hotel SeaView costs ₹3500 per night.
- Pondicherry is near Chennai.
- Pondicherry is a heritage destination.
- Hotel Heritage is located in Pondicherry.
- Hotel Heritage costs ₹3000 per night.

In Notebook 2, we represented this information as triples.

In [1]:
triples = [
    ("Chennai", "LOCATED_IN", "Tamil Nadu"),
    ("Mahabalipuram", "NEAR", "Chennai"),
    ("Mahabalipuram", "HAS_CATEGORY", "Heritage"),
    ("Marina Beach", "LOCATED_IN", "Chennai"),
    ("Chennai", "CONNECTED_TO", "Bengaluru"),
    ("Bengaluru", "CONNECTED_TO", "Mysuru"),
    ("Hotel SeaView", "LOCATED_IN", "Mahabalipuram"),
    ("Hotel SeaView", "PRICE_PER_NIGHT", 3500),
    
    ("Pondicherry", "NEAR", "Chennai"),
    ("Pondicherry", "HAS_CATEGORY", "Heritage"),
    ("Hotel Heritage", "LOCATED_IN", "Pondicherry"),
    ("Hotel Heritage", "PRICE_PER_NIGHT", 3000)
]

print("Number of triples:", len(triples))

Number of triples: 12


## Inspect the Knowledge Graph

Before querying a graph, let us see what knowledge it contains.

Each triple has the form:

    Subject → Relationship → Object

For example:

    Chennai → CONNECTED_TO → Bengaluru

This means:

- Subject = Chennai
- Relationship = CONNECTED_TO
- Object = Bengaluru

In [2]:
for subject, relationship, object_ in triples:
    print(subject, "→", relationship, "→", object_)

Chennai → LOCATED_IN → Tamil Nadu
Mahabalipuram → NEAR → Chennai
Mahabalipuram → HAS_CATEGORY → Heritage
Marina Beach → LOCATED_IN → Chennai
Chennai → CONNECTED_TO → Bengaluru
Bengaluru → CONNECTED_TO → Mysuru
Hotel SeaView → LOCATED_IN → Mahabalipuram
Hotel SeaView → PRICE_PER_NIGHT → 3500
Pondicherry → NEAR → Chennai
Pondicherry → HAS_CATEGORY → Heritage
Hotel Heritage → LOCATED_IN → Pondicherry
Hotel Heritage → PRICE_PER_NIGHT → 3000


# Part 1 — Asking a Question to the Knowledge Graph

Suppose a traveller asks:

> Where is Chennai located?

This is a natural-language question.

We first understand what the question is asking.

The important entity is:

    Chennai

The important relationship is:

    LOCATED_IN

We therefore convert the question into this graph pattern:

    Chennai → LOCATED_IN → ?

The question mark means:

> "I do not know the object. Find it for me."

This is the basic idea of graph querying.

In [3]:
for subject, relationship, object_ in triples:
    if subject == "Chennai" and relationship == "LOCATED_IN":
        print(object_)

Tamil Nadu


## What did we do?

We did not search the graph randomly.

We converted the question into a pattern:

    Chennai → LOCATED_IN → ?

Then we searched the triples for a matching pattern.

The graph answered:

    Chennai → LOCATED_IN → Tamil Nadu

Therefore:

    Chennai is located in Tamil Nadu.

### Important Idea

A Knowledge Graph allows us to ask questions in terms of
entities and relationships.

# Part 2 — Questions with an Unknown Subject

Consider the question:

> Which places are near Chennai?

Here, we know:

    Relationship = NEAR
    Object = Chennai

But we do not know the subject.

Therefore the graph pattern becomes:

    ? → NEAR → Chennai

The question mark means:

> Find all subjects that have a NEAR relationship with Chennai.

In [4]:
for subject, relationship, object_ in triples:
    if relationship == "NEAR" and object_ == "Chennai":
        print(subject)

Mahabalipuram
Pondicherry


## Notice the Difference

Question 1:

    Where is Chennai located?

Graph pattern:

    Chennai → LOCATED_IN → ?

Question 2:

    Which places are near Chennai?

Graph pattern:

    ? → NEAR → Chennai

The direction of the relationship matters.

This is one of the fundamental ideas in a Knowledge Graph.

# Part 3 — Another Graph Query

Question:

> Which cities are directly connected to Chennai?

Identify the components:

Entity:

    Chennai

Relationship:

    CONNECTED_TO

Unknown:

    ?

Graph pattern:

    Chennai → CONNECTED_TO → ?

In [5]:
for subject, relationship, object_ in triples:
    if subject == "Chennai" and relationship == "CONNECTED_TO":
        print(object_)

Bengaluru


# Part 4 — Creating a Generic Query Function

So far, we wrote separate code for every question.

But imagine having hundreds or thousands of questions.

We should create a reusable function.

We want to be able to ask:

    Find triples with a particular subject.

or:

    Find triples with a particular relationship.

or:

    Find triples with a particular object.

We can use None to mean:

    "I don't care about this part."

In [6]:
def query_graph(triples, subject=None, relationship=None, object_=None):
    
    results = []

    for s, r, o in triples:
        
        if subject is not None and s != subject:
            continue
            
        if relationship is not None and r != relationship:
            continue
            
        if object_ is not None and o != object_:
            continue
            
        results.append((s, r, o))

    return results

## Understanding the Query Function

The function has three parameters:

    subject
    relationship
    object_

If a parameter is None, we do not impose a condition on it.

For example:

    query_graph(
        triples,
        subject="Chennai",
        relationship="LOCATED_IN"
    )

means:

    Find triples where

    Subject = Chennai
    Relationship = LOCATED_IN
    Object = anything

In [7]:
result = query_graph(
    triples,
    subject="Chennai",
    relationship="LOCATED_IN"
)

result

[('Chennai', 'LOCATED_IN', 'Tamil Nadu')]

In [8]:
result = query_graph(
    triples,
    relationship="NEAR",
    object_="Chennai"
)

result

[('Mahabalipuram', 'NEAR', 'Chennai'), ('Pondicherry', 'NEAR', 'Chennai')]

# Why Did We Create a Generic Query Function?

We have separated two things:

### 1. Knowledge

The triples contain the knowledge.

### 2. Query

The query specifies what knowledge we want.

This separation becomes very important when we move to
dedicated Knowledge Graph technologies.

For example:

    Neo4j  → Cypher

    RDF    → SPARQL

In those systems, we will write formal graph queries.

For now, Python is helping us understand the concept
without introducing another technology.

# Part 5 — Moving from Querying to Traversal

Now consider a more interesting question:

> How can a traveller travel from Chennai to Mysuru
> through the connected cities?

Look at the graph:

    Chennai → Bengaluru
                  ↓
               Mysuru

There is no direct triple:

    Chennai → CONNECTED_TO → Mysuru

Instead, we have:

    Chennai → CONNECTED_TO → Bengaluru

and

    Bengaluru → CONNECTED_TO → Mysuru

Therefore, answering the question requires following
more than one relationship.

This is called:

# Multi-Hop Traversal

## What is a Hop?

A hop means following one relationship from one node to another.

For example:

    Chennai → Bengaluru

is one hop.

Then:

    Bengaluru → Mysuru

is another hop.

Therefore:

    Chennai → Bengaluru → Mysuru

requires two hops.

This is called a 2-hop path.

In [9]:
first_hop = query_graph(
    triples,
    subject="Chennai",
    relationship="CONNECTED_TO"
)

first_hop

[('Chennai', 'CONNECTED_TO', 'Bengaluru')]

In [10]:
second_hop = query_graph(
    triples,
    subject="Bengaluru",
    relationship="CONNECTED_TO"
)

second_hop

[('Bengaluru', 'CONNECTED_TO', 'Mysuru')]

## Combining the Two Hops

We can now see the path:

    Chennai
       ↓
   Bengaluru
       ↓
     Mysuru

The reasoning chain is:

    Chennai
       ↓ CONNECTED_TO
    Bengaluru
       ↓ CONNECTED_TO
    Mysuru

This is more than simply retrieving one triple.

We are traversing the graph.

# Part 6 — Building an Adjacency Graph

For traversal, it is convenient to represent connections
as an adjacency structure.

For example:

    Chennai → Bengaluru
    Bengaluru → Mysuru

We will create a dictionary:

    node → connected nodes

In [11]:
connections = {}

for subject, relationship, object_ in triples:
    
    if relationship == "CONNECTED_TO":
        
        if subject not in connections:
            connections[subject] = []
        
        connections[subject].append(object_)

connections

{'Chennai': ['Bengaluru'], 'Bengaluru': ['Mysuru']}

# Part 7 — Why Do We Need Graph Traversal Algorithms?

Our example is very small.

But imagine:

Chennai
   ↓
Bengaluru
   ↓
Mysuru
   ↓
Coorg
   ↓
Mangalore
   ↓
...

Now we cannot manually check every possible path.

We need an algorithm that can explore the graph.

Two common graph traversal techniques are:

- BFS — Breadth-First Search
- DFS — Depth-First Search

We will demonstrate BFS.

## Breadth-First Search — BFS

BFS explores a graph level by level.

Starting from Chennai:

Level 0:

    Chennai

Level 1:

    Bengaluru

Level 2:

    Mysuru

BFS is useful when we want to find a path,
especially a shortest path in an unweighted graph.

In [12]:
from collections import deque

def bfs_path(graph, start, target):
    
    queue = deque()
    
    # Store the current node and the path used to reach it
    queue.append((start, [start]))
    
    visited = set()

    while queue:
        
        current, path = queue.popleft()

        if current == target:
            return path

        if current in visited:
            continue

        visited.add(current)

        for neighbor in graph.get(current, []):
            
            if neighbor not in visited:
                queue.append(
                    (neighbor, path + [neighbor])
                )

    return None

## Finding a Path

Now let us ask:

> Is there a path from Chennai to Mysuru?

The BFS algorithm will explore the graph and
return the path if one exists.

In [13]:
path = bfs_path(
    connections,
    "Chennai",
    "Mysuru"
)

path

['Chennai', 'Bengaluru', 'Mysuru']

In [14]:
if path:
    print("Path found:")
    print(" → ".join(path))
else:
    print("No path found.")

Path found:
Chennai → Bengaluru → Mysuru


# Part 8 — Querying is Not the Same as Traversal

This distinction is important.

### Query

A query may retrieve a fact directly:

    Chennai → LOCATED_IN → Tamil Nadu

### Traversal

Traversal follows relationships:

    Chennai
       ↓
    Bengaluru
       ↓
    Mysuru

### Multi-Hop Reasoning

When we combine multiple relationships to answer a question:

    Chennai
       ↓ CONNECTED_TO
    Bengaluru
       ↓ CONNECTED_TO
    Mysuru

we are performing reasoning over a path.

Therefore:

    Query ≠ Traversal ≠ Reasoning

They are related, but conceptually different.

# Part 9 — A More Interesting Travel Question

Suppose the traveller asks:

> Find heritage destinations near Chennai.

This question has TWO conditions.

Condition 1:

    destination → NEAR → Chennai

Condition 2:

    destination → HAS_CATEGORY → Heritage

Therefore, we need to combine two sets of information.

In [15]:
near_chennai = []

for subject, relationship, object_ in triples:
    
    if relationship == "NEAR" and object_ == "Chennai":
        near_chennai.append(subject)

near_chennai

['Mahabalipuram', 'Pondicherry']

In [16]:
heritage_destinations = []

for subject, relationship, object_ in triples:
    
    if relationship == "HAS_CATEGORY" and object_ == "Heritage":
        heritage_destinations.append(subject)

heritage_destinations

['Mahabalipuram', 'Pondicherry']

## Combining the Results

We need destinations that satisfy BOTH conditions.

    Near Chennai
          AND
    Heritage

This is another form of graph-based reasoning.

We can find the intersection of the two sets.

In [17]:
result = set(near_chennai) & set(heritage_destinations)

print("Heritage destinations near Chennai:")

for place in result:
    print("-", place)

Heritage destinations near Chennai:
- Pondicherry
- Mahabalipuram


# Part 10 — Multi-Hop Travel Recommendation

Now let us make the question more realistic.

A traveller asks:

> Find affordable hotels in heritage destinations near Chennai.

This question cannot be answered from one triple.

We need to follow several relationships.

First:

    Destination → NEAR → Chennai

Second:

    Destination → HAS_CATEGORY → Heritage

Third:

    Hotel → LOCATED_IN → Destination

Fourth:

    Hotel → PRICE_PER_NIGHT → Price

Therefore, the reasoning chain looks like:

Hotel
  ↓
LOCATED_IN
  ↓
Destination
  ↓
NEAR
  ↓
Chennai

and also:

Destination
  ↓
HAS_CATEGORY
  ↓
Heritage

and finally:

Hotel
  ↓
PRICE_PER_NIGHT
  ↓
Budget

## Step 1 — Find Heritage Destinations Near Chennai

We already know how to do this.

The candidate destinations are:

    Mahabalipuram
    Pondicherry

In [18]:
candidate_destinations = []

for destination in near_chennai:
    
    if destination in heritage_destinations:
        candidate_destinations.append(destination)

candidate_destinations

['Mahabalipuram', 'Pondicherry']

## Step 2 — Find Hotels in Those Destinations

For every candidate destination, we look for:

    Hotel → LOCATED_IN → Destination

This is another traversal step.

In [19]:
hotels = []

for subject, relationship, object_ in triples:
    
    if relationship == "LOCATED_IN" and object_ in candidate_destinations:
        hotels.append((subject, object_))

hotels

[('Hotel SeaView', 'Mahabalipuram'), ('Hotel Heritage', 'Pondicherry')]

## Step 3 — Check the Hotel Price

Suppose the traveller's budget is:

    ₹4000 per night

We now check the PRICE_PER_NIGHT relationship.

We want:

    Hotel → PRICE_PER_NIGHT → Price

where:

    Price < 4000

In [20]:
budget = 4000

affordable_hotels = []

for hotel, destination in hotels:
    
    for subject, relationship, object_ in triples:
        
        if (
            subject == hotel
            and relationship == "PRICE_PER_NIGHT"
            and object_ < budget
        ):
            affordable_hotels.append(
                (hotel, destination, object_)
            )

affordable_hotels

[('Hotel SeaView', 'Mahabalipuram', 3500),
 ('Hotel Heritage', 'Pondicherry', 3000)]

# Part 11 — Putting the Reasoning Together

The answer did not come from one triple.

For example, to recommend Hotel SeaView, we followed:

    Hotel SeaView
          ↓ LOCATED_IN
    Mahabalipuram
          ↓ NEAR
    Chennai

and:

    Mahabalipuram
          ↓ HAS_CATEGORY
    Heritage

and:

    Hotel SeaView
          ↓ PRICE_PER_NIGHT
         ₹3500

The complete reasoning chain is therefore:

Hotel SeaView
    ↓
Mahabalipuram
    ↓
near Chennai
    ↓
Heritage destination

and

Hotel SeaView
    ↓
₹3500
    ↓
within ₹4000 budget

This is the power of connecting pieces of knowledge.

# Part 12 — Create a Reusable Travel Query

Instead of writing separate code every time,
let us create a function.

Question:

> Find affordable hotels in heritage destinations
> near a given city.

In [21]:
def find_travel_options(triples, city, budget):
    
    # Step 1: Find places near the city
    nearby_places = set()
    
    for subject, relationship, object_ in triples:
        if relationship == "NEAR" and object_ == city:
            nearby_places.add(subject)

    # Step 2: Keep only heritage places
    heritage_places = set()
    
    for subject, relationship, object_ in triples:
        if (
            relationship == "HAS_CATEGORY"
            and object_ == "Heritage"
            and subject in nearby_places
        ):
            heritage_places.add(subject)

    # Step 3: Find hotels in those places
    hotels = []
    
    for subject, relationship, object_ in triples:
        if (
            relationship == "LOCATED_IN"
            and object_ in heritage_places
        ):
            hotels.append((subject, object_))

    # Step 4: Check hotel prices
    results = []
    
    for hotel, destination in hotels:
        
        for subject, relationship, object_ in triples:
            
            if (
                subject == hotel
                and relationship == "PRICE_PER_NIGHT"
                and object_ < budget
            ):
                results.append({
                    "hotel": hotel,
                    "destination": destination,
                    "price": object_
                })

    return results

In [22]:
options = find_travel_options(
    triples,
    city="Chennai",
    budget=4000
)

for option in options:
    print(option)

{'hotel': 'Hotel SeaView', 'destination': 'Mahabalipuram', 'price': 3500}
{'hotel': 'Hotel Heritage', 'destination': 'Pondicherry', 'price': 3000}


# Part 13 — How Did We Solve the Question?

Let us step back.

The traveller gave us a natural-language question:

> Find affordable hotels in heritage destinations near Chennai.

We did NOT directly search for the answer.

We decomposed the question.

### Step 1 — Identify the main entity

    Chennai

### Step 2 — Identify the constraints

    NEAR
    Heritage
    Hotel
    Price

### Step 3 — Convert them into graph patterns

    ? → NEAR → Chennai

    ? → HAS_CATEGORY → Heritage

    Hotel → LOCATED_IN → ?

    Hotel → PRICE_PER_NIGHT → ?

### Step 4 — Query the graph

### Step 5 — Traverse relationships

### Step 6 — Combine the results

### Step 7 — Apply the budget condition

### Step 8 — Produce the answer

This is the basic thinking required for Knowledge Graph reasoning.

# Part 14 — An Important Question

So far, WE identified the entities and relationships.

For example:

Question:

    Find affordable hotels near Chennai.

We manually identified:

Entity:

    Chennai

Concepts / entities:

    hotels

Relationship:

    NEAR

Property:

    PRICE_PER_NIGHT

But now ask:

> Can a computer identify these automatically from the sentence?

For example:

    "Find affordable hotels near Chennai."

Could a machine automatically identify:

    Entity → Chennai

    Relationship → NEAR

    Entity Type → Hotel

    Constraint → Affordable

This is a much deeper problem.

It leads us toward:

    Natural Language Processing
             ↓
    Information Extraction
             ↓
    Entity Recognition
             ↓
    Relationship Extraction
             ↓
    Triple Generation
             ↓
    Knowledge Graph Construction

## Important Pedagogical Decision

We will NOT solve automatic entity and relationship extraction yet.

Why?

Because students should first understand:

    What is an entity?
    What is a relationship?
    What is a triple?
    What is a graph pattern?
    What is a query?
    What is traversal?
    What is multi-hop reasoning?

Only after these ideas are clear should we ask:

> How can a machine automatically construct these triples
> from natural language?

That topic can later connect Knowledge Graphs with
NLP and LLMs.

# Exercise 1 — Convert Questions into Graph Patterns

For each question, identify:

1. Entity
2. Relationship
3. Unknown part
4. Graph pattern

### Question A

Where is Mahabalipuram located?

### Question B

Which places are near Chennai?

### Question C

Which hotels are located in Pondicherry?

### Question D

Which cities are connected to Bengaluru?

Do NOT write Python first.

Think about the graph pattern first.

# Exercise 2 — Multi-Hop Thinking

Consider:

    Chennai
       ↓
    Bengaluru
       ↓
    Mysuru

Question:

> Can a traveller reach Mysuru from Chennai through
> connected cities?

Identify:

1. Starting entity
2. Target entity
3. Relationship
4. Number of hops
5. Path

Again, think about the graph before writing code.

# Exercise 3 — Add New Travel Knowledge

Add the following facts:

- Chennai is connected to Hyderabad.
- Hyderabad is connected to Bengaluru.
- Hyderabad is a heritage destination.
- Charminar Hotel is located in Hyderabad.
- Charminar Hotel costs ₹2800 per night.

Represent each fact as a triple.

In [23]:
new_triples = [
    ("Chennai", "CONNECTED_TO", "Hyderabad"),
    ("Hyderabad", "CONNECTED_TO", "Bengaluru"),
    ("Hyderabad", "HAS_CATEGORY", "Heritage"),
    ("Charminar Hotel", "LOCATED_IN", "Hyderabad"),
    ("Charminar Hotel", "PRICE_PER_NIGHT", 2800)
]

triples.extend(new_triples)

print("Total number of triples:", len(triples))

Total number of triples: 17


# Exercise 4 — Try Your Own Questions

Now formulate three travel questions.

For each question:

1. Write the natural-language question.
2. Identify the entities.
3. Identify the relationship(s).
4. Draw the graph pattern.
5. Decide whether it requires:
   - Direct query
   - One-hop traversal
   - Multi-hop traversal
6. Write Python code to answer it.

Example:

Question:

    Which hotels are in heritage destinations?

Possible reasoning:

    Hotel
      ↓ LOCATED_IN
    Destination
      ↓ HAS_CATEGORY
    Heritage

# Final Reflection

Before moving to the next notebook, think about these questions.

### 1.

Why is:

    Chennai → LOCATED_IN → Tamil Nadu

a direct query?

### 2.

Why is:

    Chennai → Bengaluru → Mysuru

a multi-hop traversal?

### 3.

Why can a Knowledge Graph answer questions involving
multiple relationships?

### 4.

Why did we first identify entities and relationships
manually?

### 5.

What would we need if we wanted a computer to automatically
extract entities and relationships from:

    "Find affordable hotels near Chennai."

These questions lead naturally to the next stages
of Knowledge Graph development.

# Key Takeaways

In this notebook we learned:

### 1. Graph Query

Ask the graph for a particular pattern.

### 2. Graph Pattern

Represent a question as:

    Subject → Relationship → Object

with ? for an unknown component.

### 3. Traversal

Follow relationships from one node to another.

### 4. Hop

One relationship traversal is one hop.

### 5. Multi-Hop

Following multiple relationships:

    A → B → C

### 6. Reasoning

Combine information from multiple triples
to derive an answer.

### 7. Question-to-Graph Thinking

    Natural-language Question
              ↓
       Identify Entities
              ↓
     Identify Relationships
              ↓
         Graph Pattern
              ↓
            Query
              ↓
         Traversal
              ↓
          Reasoning
              ↓
            Answer

# Notebook 3 → Notebook 4

We have so far created the Knowledge Graph manually in Python.

But in a real application, knowledge may arrive as data files.

For example:

    travel_triples.csv

containing:

    Subject,Relationship,Object

The next question is:

> How can we build a Knowledge Graph from a CSV file?

In Notebook 4 we will learn:

    CSV Data
       ↓
    Read Data
       ↓
    Validate Data
       ↓
    Convert to Triples
       ↓
    Build Knowledge Graph
       ↓
    Query
       ↓
    Traverse

We will continue using the same Travel Planning problem.

                NOTEBOOK 3

         Natural-language question
                    ↓
          Identify what is known
                    ↓
           Identify what is unknown
                    ↓
             Graph pattern
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
       Direct Query       Traversal
          ↓                   ↓
       1 triple          Multiple triples
                              ↓
                         Multi-hop
                              ↓
                         Reasoning
                              ↓
                      Travel recommendation
                              ↓
                 "Can machine identify these?"
                              ↓
                    NLP / Information
                       Extraction
The KG is not only about storing knowledge. The real intelligence comes from being able to represent a question as a graph pattern and follow relationships to construct an answer.